# Kaggle copy vs official ARCADE (Zenodo) checksum check

**Needs: Internet ON (Settings > Internet), dataset `nirmalgaud/arcade-dataset` added as input. No GPU. ~5 min.** Output: `checksum_check.csv` - send it back (or copy the printed table).

In [1]:
import os, hashlib, zipfile, subprocess, collections
from pathlib import Path
import pandas as pd
OFFICIAL_MD5 = "c5b1973ade06f7dff210f878161e1a76"   # Zenodo record 10390295, arcade.zip, 451.6 MB
URL = "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
W = Path("/kaggle/working"); W.mkdir(exist_ok=True)
Z = W / "arcade.zip"
if not Z.exists():
    subprocess.run(["wget", "-q", "-O", str(Z), URL], check=True)
h = hashlib.md5()
with open(Z, "rb") as f:
    for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
print("official zip md5:", h.hexdigest(), "| expected:", OFFICIAL_MD5, "| OK" if h.hexdigest() == OFFICIAL_MD5 else "| MISMATCH")
OFF = W / "official"; OFF.mkdir(exist_ok=True)
with zipfile.ZipFile(Z) as zf: zf.extractall(OFF)
def md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest()
def hashes(root, part):
    out = {}
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in (".png", ".jpg", ".jpeg") and part in str(p).lower().replace("\\", "/"):
            sp = [s for s in ("train", "val", "test") if f"/{s}/" in str(p).lower().replace("\\", "/")]
            out[(sp[0] if sp else "?", p.name)] = md5(p)
    return out
kag = [p.parent.parent.parent for p in Path("/kaggle/input").rglob("train.json") if "stenosis" in str(p).lower()][0].parent
rows = []
for part in ("stenosis", "syntax"):
    A = hashes(kag, part); B = hashes(OFF, part)
    same = sum(1 for k in A if k in B and A[k] == B[k]); onlyk = [k for k in A if k not in B]; diff = [k for k in A if k in B and A[k] != B[k]]
    rows.append(dict(part=part, kaggle_images=len(A), official_images=len(B), identical_by_name_and_md5=same, only_in_kaggle=len(onlyk), same_name_different_bytes=len(diff)))
R = pd.DataFrame(rows); print(R.to_string(index=False)); R.to_csv(W / "checksum_check.csv", index=False)
print("\nKaggle root:", kag)


official zip md5: c5b1973ade06f7dff210f878161e1a76 | expected: c5b1973ade06f7dff210f878161e1a76 | OK
    part  kaggle_images  official_images  identical_by_name_and_md5  only_in_kaggle  same_name_different_bytes
stenosis           1500             1500                       1500               0                          0
  syntax           1500             1500                       1500               0                          0

Kaggle root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade
